# Clustering burn calculation

In [3]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from shapely import wkt 
from google.cloud import bigquery
from shapely.wkt import loads as load_wkt
from shapely.prepared import prep
from tqdm import tqdm

In [5]:
# Optional configurations
# warnings.filterwarnings('ignore')
pd.set_option("display.max_columns", None)

client = bigquery.Client( project = 'bi-team-400508')
Awb= ''' with awb_data as (
select sg.order_date,sg.rider_id,sg.hub_id,sg.pincode,sg.pincode_category,sg.order_id as fwd_del_awb_number,edp.delivery_latitude as lat,edp.delivery_longitude as long,ROW_NUMBER() OVER (PARTITION BY sg.rider_id ORDER BY edp.update_timestamp) AS row_num
from data-warehousing-391512.smaug_dataengine.data_engine_orderleveldata sg
left join data-warehousing-391512.ecommerce.ecommerce_deliveryrequest edr on edr.awb_number = sg.order_id and edr.last_updated > current_date - interval 30 day
left join data-warehousing-391512.ecommerce.ecommerce_deliveryrequestproof edp on edr.id = edp.delivery_request_id and edp.update_timestamp > current_date - interval 30 day
where sg.order_date > current_date - interval 30 day and sg.order_category = 1 and ecom_request_type in (1) and sg.order_status in (1) and sg.order_tag in (0, 1, 14) and edr.client_id not in (5,18,60,61,67,68,102,354,552,557,715,67,818,862,875,11,996,1579,1575,1819,2063,2253)
and sg.hub_id in (15344)

union all
select sg.order_date,sg.rider_id,sg.hub_id,sg.pincode,sg.pincode_category,sg.order_id as fwd_del_awb_number,epp.pickup_latitude as lat,epp.pickup_longitude as long,ROW_NUMBER() OVER (PARTITION BY sg.rider_id ORDER BY epp.update_timestamp) AS row_num
from data-warehousing-391512.smaug_dataengine.data_engine_orderleveldata sg
left join data-warehousing-391512.ecommerce.pickup_pickuprequestproof epp on sg.order_id = epp.pickup_request_id and epp.update_timestamp > current_date - interval 30 day
where sg.order_date > current_date - interval 30 day and sg.order_category = 1 and ecom_request_type in (5) and sg.order_status in (2,3) and sg.order_tag in (0,1,14) 
and sg.hub_id in (15344))


Select order_date,rider_id, eh.name as hub, ad.pincode as pincode,CONCAT(CAST("P" AS STRING), ad.pincode_category) AS payment_category,fwd_del_awb_number,
COALESCE(lat, FIRST_VALUE(lat) OVER (PARTITION BY rider_id ORDER BY row_num ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)) AS lat,COALESCE(long, FIRST_VALUE(long) OVER (PARTITION BY rider_id ORDER BY row_num ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)) AS long
FROM awb_data ad
left join data-warehousing-391512.ecommerce.ecommerce_hub eh on ad.hub_id = eh.id
;
'''


print("Running BigQuery...")
Awn_number_with_latlong = client.query(Awb).to_dataframe()
print("Query complete.")
Awn_number_with_latlong

E:\Ravindra Kumar\Anaconda3\Lib\site-packages\google\auth\_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


Running BigQuery...


E:\Ravindra Kumar\Anaconda3\Lib\site-packages\google\cloud\bigquery\table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Query complete.


,order_date,rider_id,hub,pincode,payment_category,fwd_del_awb_number,lat,long
0,2026-04-09,25933593,WRR_ShegaonBK_SCC,442914,P4,R1995713160FPL,20.2560383,79.0152583
1,2026-05-01,25933593,WRR_ShegaonBK_SCC,442906,P1,SF3306947197FPL,20.2560383,79.0152583
2,2026-04-09,25933593,WRR_ShegaonBK_SCC,442906,P1,R1673643537F,20.255897,79.015377
3,2026-04-14,25933593,WRR_ShegaonBK_SCC,442906,P1,SF3240953319FPL,20.2560383,79.0152583
4,2026-04-09,25933593,WRR_ShegaonBK_SCC,442906,P1,R1673529050F,20.25617,79.0152283
...,...,...,...,...,...,...,...,...
4300,2026-05-07,25933593,WRR_ShegaonBK_SCC,442906,P1,SF3343387989FPL,20.34841,79.15292
4301,2026-05-07,25933593,WRR_ShegaonBK_SCC,442906,P1,SF3340289537FPL,20.2134566,79.0680945
4302,2026-05-07,25933593,WRR_ShegaonBK_SCC,442906,P1,SF3344180776FPL,20.2131134,79.0688292
4303,2026-05-07,25933593,WRR_ShegaonBK_SCC,442906,P1,SF3333903832FPL,20.328065,79.1426317


In [7]:
import pandas as pd
from shapely.wkt import loads as load_wkt
from shapely.geometry import Point
from shapely.prepared import prep
from tqdm import tqdm


def load_clusters(csv_file):
    df = pd.read_csv(csv_file)

    clusters = []
    skipped = []

    for idx, row in df.iterrows():
        try:
            wkt_str = str(row['WKT']).strip()

            # skip empty geometries
            if 'POLYGON ()' in wkt_str:
                skipped.append((idx, row['name'], 'Empty polygon inside geometry'))
                continue

            # skip geometry collections (or you can extract them if needed)
            if wkt_str.startswith('GEOMETRYCOLLECTION'):
                skipped.append((idx, row['name'], 'GeometryCollection not supported'))
                continue

            polygon = load_wkt(wkt_str)

            if polygon.is_empty:
                skipped.append((idx, row['name'], 'Parsed empty geometry'))
                continue

            clusters.append({
                'prepared': prep(polygon),
                'polygon': polygon,
                'name': row['name'],
                'description': row['description']
            })

        except Exception as e:
            skipped.append((idx, row['name'], str(e)))

    if skipped:
        print(f"\n⚠️ Skipped {len(skipped)} invalid cluster rows:")
        for s in skipped:
            print(f"Row {s[0]} | Name: {s[1]} | Reason: {s[2]}")

    print(f"\n✅ Loaded {len(clusters)} clusters successfully\n")

    return clusters


def get_cluster_for_point(lat, lon, clusters):
    if pd.isnull(lat) or pd.isnull(lon):
        return None, None

    point = Point(lon, lat)

    for cluster in clusters:
        if cluster['prepared'].contains(point):
            return cluster['name'], cluster['description']

    return None, None


def process_awb_data(awb_df, clusters):
    results = []

    for row in tqdm(awb_df.itertuples(index=False), total=len(awb_df)):
        lat = row.lat
        lon = row.long
        pincode = row.pincode

        name, description = get_cluster_for_point(lat, lon, clusters)

        if not name:
            pincode_map = {
                580011: "C4",
                203209: "C8",
                282009: "C6",
                584128: "C2",
                110074: "C2",
                800001: "C0"
            }

            if pincode in pincode_map:
                name = "Previous mapping"
                description = pincode_map[pincode]

        results.append({
            'order_date': row.order_date,
            'awb_number': row.fwd_del_awb_number,
            'rider_id': row.rider_id,
            'pincode': pincode,
            'payment_category': row.payment_category,
            'hub': row.hub,
            'lat': lat,
            'long': lon,
            'cluster_name': name,
            'description': description
        })

    return pd.DataFrame(results)


if __name__ == "__main__":
    clusters = load_clusters("Live5.csv")
    final_result_df = process_awb_data(Awn_number_with_latlong, clusters)
    print(final_result_df.head())


✅ Loaded 11 clusters successfully



100%|██████████| 4305/4305 [00:00<00:00, 12136.69it/s]

   order_date       awb_number  rider_id  pincode payment_category  \
0  2026-04-09   R1995713160FPL  25933593   442914               P4   
1  2026-05-01  SF3306947197FPL  25933593   442906               P1   
2  2026-04-09     R1673643537F  25933593   442906               P1   
3  2026-04-14  SF3240953319FPL  25933593   442906               P1   
4  2026-04-09     R1673529050F  25933593   442906               P1   

                 hub         lat        long cluster_name description  
0  WRR_ShegaonBK_SCC  20.2560383  79.0152583     442906_L          C1  
1  WRR_ShegaonBK_SCC  20.2560383  79.0152583     442906_L          C1  
2  WRR_ShegaonBK_SCC   20.255897   79.015377     442906_L          C1  
3  WRR_ShegaonBK_SCC  20.2560383  79.0152583     442906_L          C1  
4  WRR_ShegaonBK_SCC    20.25617  79.0152283     442906_L          C1  


In [9]:
# # Load cluster polygons from CSV
# def load_clusters(csv_file):
#     df = pd.read_csv(csv_file)
#     clusters = []
#     for _, row in df.iterrows():
#         polygon = load_wkt(row['WKT'])
#         clusters.append({
#             'prepared': prep(polygon),
#             'polygon': polygon,
#             'name': row['name'],
#             'description': row['description']
#         })
#     return clusters

# # Identify the cluster for a given point
# def get_cluster_for_point(lat, lon, clusters):
#     if pd.isnull(lat) or pd.isnull(lon):
#         return None, None
#     point = Point(lon, lat)
#     for cluster in clusters:
#         if cluster['prepared'].contains(point):
#             return cluster['name'], cluster['description']
#     return None, None

# # Process AWB data and assign clusters
# def process_awb_data(awb_df, clusters):
#     results = []
#     tqdm.pandas(desc="Processing AWBs")

#     for _, row in tqdm(awb_df.iterrows(), total=len(awb_df)):
#         lat = row['lat']
#         lon = row['long']
#         pincode = row['pincode']
#         name, description = get_cluster_for_point(lat, lon, clusters)

#         # Fallback using pincode if no cluster match
#         if not name:
#             pincode_map = {
#                 580011: "C4",
#                 203209: "C8",
#                 282009: "C6",
#                 584128: "C2",
#                 110074: "C2",
#                 800001: "C0"
#             }
#             if pincode in pincode_map:
#                 name = "Previous mapping"
#                 description = pincode_map[pincode]

#         results.append({
#             'order_date': row['order_date'],
#             'awb_number': row['fwd_del_awb_number'],
#             'rider_id': row['rider_id'],
#             'pincode': pincode,
#             'payment_category': row['payment_category'],
#             'hub': row['hub'],
#             'lat': lat,
#             'long': lon,
#             'cluster_name': name,
#             'description': description
#         })

#     result_df = pd.DataFrame(results)
#     # result_df.to_csv("Awb_with_cluster_info.csv", index=False)
#     # print("Saved to 'Awb_with_cluster_info.csv'")
#     return result_df  # ✅ Added return

# # Main execution
# if __name__ == "__main__":
#     clusters = load_clusters("Live5.csv",)
#     final_result_df = process_awb_data(Awn_number_with_latlong, clusters)
#     print(final_result_df)# ✅ Correct usage

In [11]:
final_result_df.to_csv("final_result_df.csv", index=False)
print ("final_result is saved")

final_result is saved


In [13]:
# Mapping and calculations
payment_mapping = {
    "P1": 0, "РЗ1": 0.5, "P2": 1, "РЗ2": 1.5, "P3": 2, "РЗЗ": 2.5, "Р34": 3, "РЗ5": 3.5, "P4": 4,
    "РЗ6": 4.5, "РЗ7": 5, "РЗ8": 5.5, "P5": 6, "РЗ9": 6.5, "P40": 7, "Р41": 7.5, "P6": 8, "P7": 9,
    "P8": 10, "P9": 10, "P10": 10, "P11": 11, "P12": 12, "P13": 13, "P14": 14, "P15": 15, "P16": 16,
    "P17": 17, "P18": 18, "P19": 19, "P20": 20, "P21": 21, "P22": 22, "P23": 23, "P24": 24, "P25": 25,
    "P26": 26, "P27": 27, "P28": 28, "P29": 29, "P30": 30
}

description_mapping = {
    "C1": 0, "C2": 0.5, "C3": 1, "C4": 1.5, "C5": 2, "C6": 2.5, "C7": 3, "C8": 3.5, "C9": 4,
    "C10": 4.5, "C11": 5, "C12": 6, "C13": 7, "C14": 8, "C15": 9, "C16": 10, "C17": 11, "C18": 12,
    "C19": 13, "C20": 15
}

# Perform mapping and calculations
final_result_df["Pin_Pay"] = final_result_df["payment_category"].map(payment_mapping).fillna(0)
final_result_df["Clustering_payout"] = final_result_df["description"].map(description_mapping).fillna(final_result_df["Pin_Pay"])
final_result_df["P & L"] = final_result_df["Pin_Pay"] - final_result_df["Clustering_payout"]
final_result_df["Saving"] = final_result_df["P & L"].apply(lambda x: x if x > 0 else 0)
final_result_df["Burning"] = final_result_df["P & L"].apply(lambda x: -x if x < 0 else 0)

# Remove rows where all financials are 0
columns_to_check = ["Pin_Pay", "Clustering_payout", "Saving", "Burning", "P & L"]
final_result_df = final_result_df[~(final_result_df[columns_to_check] == 0).all(axis=1)]

# Reset index
final_result_df.reset_index(drop=True, inplace=True)

# Save and return
# final_result_df.to_csv("Awb_with_cluster_info.csv", index=False)
# print("Saved to 'Awb_with_cluster_info.csv'")
final_result_df


,order_date,awb_number,rider_id,pincode,payment_category,hub,lat,long,cluster_name,description,Pin_Pay,Clustering_payout,P & L,Saving,Burning
0,2026-04-09,R1995713160FPL,25933593,442914,P4,WRR_ShegaonBK_SCC,20.2560383,79.0152583,442906_L,C1,4.0,0.0,4.0,4.0,0.0
1,2026-04-10,R1998683813FPL,25933593,442906,P1,WRR_ShegaonBK_SCC,20.2480845,79.1801052,442906_C,C7,0.0,3.0,-3.0,0.0,3.0
2,2026-04-10,R1998669581FPL,25933593,442906,P1,WRR_ShegaonBK_SCC,20.2372949,79.1900747,442906_C,C7,0.0,3.0,-3.0,0.0,3.0
3,2026-04-08,SF3228193783FPL,25933593,442906,P1,WRR_ShegaonBK_SCC,20.2378579,79.1307695,442906_B,C5,0.0,2.0,-2.0,0.0,2.0
4,2026-04-10,R1993870776FPL,25933593,442906,P1,WRR_ShegaonBK_SCC,20.2008396,79.2379691,442906_D,C9,0.0,4.0,-4.0,0.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3980,2026-05-07,SF2977083436F,25933593,442906,P1,WRR_ShegaonBK_SCC,20.3821783,79.20174,442906_D,C9,0.0,4.0,-4.0,0.0,4.0
3981,2026-05-07,SF2976662487F,25933593,442906,P1,WRR_ShegaonBK_SCC,20.4280117,79.1980783,442906_E,C11,0.0,5.0,-5.0,0.0,5.0
3982,2026-05-07,SF3343387989FPL,25933593,442906,P1,WRR_ShegaonBK_SCC,20.34841,79.15292,442906_C,C7,0.0,3.0,-3.0,0.0,3.0
3983,2026-05-07,SF3333903832FPL,25933593,442906,P1,WRR_ShegaonBK_SCC,20.328065,79.1426317,442906_C,C7,0.0,3.0,-3.0,0.0,3.0


In [15]:

# Create Pivot Table
pivot_table = final_result_df.pivot_table(
    index='hub',
    values=['Pin_Pay', 'Clustering_payout', 'Saving', 'Burning', 'P & L'],
    aggfunc='sum',
    margins=True,
    margins_name='Grand Total'
)

# Reset index to remove multi-index header issues
pivot_table = pivot_table.reset_index()

# Rename Columns
report = pivot_table.rename(columns={
    'Pin_Pay': 'Expt_Pincode_Pay',
    'Clustering_payout': 'Cluster_Payout',
    'hub': 'hub'
})

# Ensure numeric columns are numeric
numeric_cols = ['Expt_Pincode_Pay', 'Cluster_Payout', 'Saving', 'Burning', 'P & L']
report[numeric_cols] = report[numeric_cols].apply(pd.to_numeric, errors='coerce')

# Add percentage columns
report['P & L %'] = (report['P & L'] / report['Expt_Pincode_Pay']) * 100 

# Round percentage columns
report['P & L %'] = report['P & L %'].round(2) 

# Split Grand Total and sort the rest
grand_total = report[report['hub'] == 'Grand Total']
report_wo_total = report[report['hub'] != 'Grand Total']
report_sorted = pd.concat([grand_total, report_wo_total.sort_values(by='P & L', ascending=False)])

# Final column order
final_cols = ['hub'] + numeric_cols + ['P & L %']
report_sorted = report_sorted[final_cols]

# Set index and remove index name to avoid blank header
report_sorted = report_sorted.set_index(report_sorted.columns[0])
report_sorted=report_sorted.rename_axis(index= None)
report_sorted

# Styling function
def highlight_profit_loss(val):
    if isinstance(val, (int, float)):
        if val > 0:
            return 'color: green'
        elif val < 0:
            return 'color: red'
    return ''

# Apply styling
styled_report = (
    report_sorted.style
    .applymap(highlight_profit_loss, subset=['P & L', 'P & L %'])
    .format({
        'Expt_Pincode_Pay': '{:,.0f}',
        'Cluster_Payout': '{:,.0f}',
        'Saving': '{:,.0f}',
        'Burning': '{:,.0f}',
        'P & L': '{:,.0f}',
        'P & L %': '{:.2f}%'
    })
    .set_properties(**{'text-align': 'center'})
    .set_table_styles([
        {'selector': 'th', 'props': [
            ('text-align', 'center'),
            ('font-weight', 'bold'),
            ('background-color', '#99ccff')
        ]}
    ])
)

Pivot_Table = styled_report 
Pivot_Table

C:\Users\RAVINDRA KUMAR\AppData\Local\Temp\ipykernel_26848\989190712.py:56: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  .applymap(highlight_profit_loss, subset=['P & L', 'P & L %'])


,Expt_Pincode_Pay,Cluster_Payout,Saving,Burning,P & L,P & L %
Grand Total,"1,720","17,045",798,"16,123","-15,325",-890.99%
WRR_ShegaonBK_SCC,"1,720","17,045",798,"16,123","-15,325",-890.99%
